# training error — Python demo

Numerical companion to the entry [training error](https://dictionaryofml.org/terms/trainerr.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

The entry's weather narrative, carried out on the record itself: the daily minimum and maximum air temperature at Krems an der Donau for all 366 days of 2024 (GeoSphere Austria, station 3805, dataset klima-v2-1d). Each day is one data point: its feature is the morning minimum, its label the maximum of the day.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/trainerr.py`](https://dictionaryofml.org/terms/trainerr.py); CC BY 4.0.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "trainerr.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""
trainerr.py -- numerical companion to the entry 'training error'.

The entry's weather narrative, carried out on the record itself: the daily
minimum and maximum air temperature at Krems an der Donau for all 366 days
of 2024 (GeoSphere Austria, station 3805, dataset klima-v2-1d).  Each day
is one data point: its feature is the morning minimum, its label the
maximum of the day.

Two examples.  The first is the picture the entry opens with: six days
spread over the range of morning minima, four of them a training set and
two held back.  A quadratic fitted to the four training days incurs an
average squared error of 0.24 on them and of 25.43 on the two held-back
days, so the training error says almost nothing about the two days the
curve never saw.

The second is larger: every ninth day of the year, forty days, is the
training set, and the remaining 326 days are held back.  A straight line
fitted to the forty days incurs an average squared error of 18.96 on them
and of 19.36 on the held-back days, a gap of two percent: a line has two
parameters, so there is little room to follow the noise of forty days.
Raising the degree of the polynomial widens the gap without bound.  The
training error falls monotonically from 71.78 for the constant to 11.76 at
degree twelve, while the held-back error is smallest around degree four,
at 19.27, and reaches 364764 at degree twelve, where the polynomial swings
wildly between the training days.

Deterministic: no randomness (every fit is the closed-form least-squares
solution, and the two subsets are fixed strides through the year).
Self-contained: numpy + matplotlib only (stdlib urllib for the download).

Blocks
------
[B-fetch]   Download the daily minimum and maximum temperature at Krems
            for 2024; write the 366 records to trainerr_weather.csv and
            check them against the archive.
[B-picture] The six days of the entry's opening figure: four training
            days, two held back, and the quadratic fitted to the four.
            Check that the training error is far below the error on the
            held-back days, and write the points, the curve and the
            prediction errors for the figure.
[B-def]     The training error of the line fitted to the forty training
            days: the average of the forty squared errors, and no line
            attains a smaller one.
[B-degree]  Polynomials of degree 0 to 12 fitted to the same forty days:
            the training error never increases with the degree, while the
            error on the 326 held-back days is smallest at a low degree
            and then explodes.

Outputs
-------
trainerr_weather.csv      : date, tmin, tmax of the 366 downloaded days
trainerr_picture_train.csv: tmin, tmax of the four training days
trainerr_picture_val.csv  : tmin, tmax of the two held-back days
trainerr_picture_curve.csv: tmin, tmax along the fitted quadratic
trainerr_picture_err.csv  : tmin, curve, errplus, errminus -- the vertical
                            gap between each day and the curve
trainerr_degree.csv       : degree, train, heldout -- the two average
                            losses of the polynomial of each degree
trainerr.png              : preview (checking only)
"""

import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")


def avg_sqerr(x, y, coef):
    return float(np.mean((y - np.polyval(coef, x)) ** 2))

**[B-fetch]** Download the daily minimum and maximum temperature at Krems for 2024; write the 366 records to trainerr_weather.csv and check them against the archive.

In [ ]:
print("[B-fetch] the record")
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1d?parameters=tlmin,tlmax&station_ids=3805"
       "&start=2024-01-01&end=2024-12-31")
with urllib.request.urlopen(URL, timeout=120) as resp:
    payload = json.load(resp)
params = payload["features"][0]["properties"]["parameters"]
dates = [stamp[:10] for stamp in payload["timestamps"]]
tmin = np.array(params["tlmin"]["data"], dtype=float)
tmax = np.array(params["tlmax"]["data"], dtype=float)
with open(OUT_DIR / "trainerr_weather.csv", "w") as f:
    f.write("date,tmin,tmax\n")
    for day, lo, hi in zip(dates, tmin, tmax):
        f.write(f"{day},{lo},{hi}\n")
print(f"    {len(dates)} days, morning minimum from {tmin.min():.1f} to "
      f"{tmin.max():.1f}, maximum from {tmax.min():.1f} to {tmax.max():.1f}")
check("[B-fetch] 366 days downloaded for 2024", len(dates) == 366)
check("[B-fetch] the record matches the archive (Jan 1: 2.6 to 9.6)",
      (dates[0], tmin[0], tmax[0]) == ("2024-01-01", 2.6, 9.6))
check("[B-fetch] no day is missing a temperature",
      not np.isnan(tmin).any() and not np.isnan(tmax).any())

**[B-picture]** The six days of the entry's opening figure: four training days, two held back, and the quadratic fitted to the four. Check that the training error is far below the error on the held-back days, and write the points, the curve and the prediction errors for the figure.

In [ ]:
print("\n[B-picture] the six days of the entry's opening figure")
by_feature = np.argsort(tmin, kind="stable")
QUANTILES = (0.05, 0.22, 0.40, 0.58, 0.76, 0.95)
six = np.array([by_feature[int(q * (len(tmin) - 1))] for q in QUANTILES])
pic_train, pic_val = six[[0, 2, 3, 5]], six[[1, 4]]   # the 2nd and 5th held back
quad = np.polyfit(tmin[pic_train], tmax[pic_train], 2)
err_train = avg_sqerr(tmin[pic_train], tmax[pic_train], quad)
err_val = avg_sqerr(tmin[pic_val], tmax[pic_val], quad)
for label, group in (("training", pic_train), ("held back", pic_val)):
    for d in group:
        print(f"    {label:>9}: {dates[d]}  morning minimum {tmin[d]:5.1f}, "
              f"maximum {tmax[d]:5.1f}, curve {np.polyval(quad, tmin[d]):5.1f}")
print(f"    average squared error {err_train:.2f} on the four training days, "
      f"{err_val:.2f} on the two held-back days")
check("[B-picture] six days spread over the range of morning minima",
      len(np.unique(six)) == 6 and tmin[six].max() - tmin[six].min() > 20.0)
check("[B-picture] the quadratic incurs a visible but small error on the "
      "training days", 0.1 < err_train < 3.0)
check("[B-picture] its error on the two held-back days is more than ten "
      "times larger", err_val > 10.0 * err_train)

grid = np.linspace(tmin[six].min() - 1.5, tmin[six].max() + 1.5, 200)
np.savetxt(OUT_DIR / "trainerr_picture_curve.csv",
           np.stack([grid, np.polyval(quad, grid)], 1), delimiter=",",
           header="tmin,tmax", comments="", fmt="%.3f")
for name, group in (("train", pic_train), ("val", pic_val)):
    np.savetxt(OUT_DIR / f"trainerr_picture_{name}.csv",
               np.stack([tmin[group], tmax[group]], 1), delimiter=",",
               header="tmin,tmax", comments="", fmt="%.2f")
with open(OUT_DIR / "trainerr_picture_err.csv", "w") as f:
    f.write("tmin,curve,errplus,errminus\n")
    for d in np.sort(six):
        on_curve = float(np.polyval(quad, tmin[d]))
        gap = float(tmax[d]) - on_curve
        f.write(f"{tmin[d]:.2f},{on_curve:.3f},{max(gap, 0.0):.3f},"
                f"{max(-gap, 0.0):.3f}\n")

**[B-def]** The training error of the line fitted to the forty training days: the average of the forty squared errors, and no line attains a smaller one.

In [ ]:
print("\n[B-def] the training error of a line fitted by ERM")
train = np.arange(len(tmin))[::9][:40]           # every ninth day, forty of them
held = np.setdiff1d(np.arange(len(tmin)), train)
line = np.polyfit(tmin[train], tmax[train], 1)
losses = (tmax[train] - np.polyval(line, tmin[train])) ** 2
trainerr = avg_sqerr(tmin[train], tmax[train], line)
print(f"    line: maximum = {line[0]:.3f} * minimum + {line[1]:.3f}")
print(f"    training error {trainerr:.2f} = average of {len(train)} squared "
      f"errors (smallest {losses.min():.2f}, largest {losses.max():.2f}); "
      f"on the {len(held)} held-back days {avg_sqerr(tmin[held], tmax[held], line):.2f}")
check("[B-def] the training error is the average of the per-day losses",
      np.isclose(trainerr, float(np.mean(losses)), atol=1e-12))
gen = np.random.default_rng(1)
others = line + gen.normal(0.0, [0.05, 0.5], size=(2000, 2))
worse = min(avg_sqerr(tmin[train], tmax[train], c) for c in others)
print(f"    2000 other lines (slope and offset perturbed): smallest training "
      f"error {worse:.2f}")
check("[B-def] no other line has a smaller training error: ERM delivers the "
      "minimum", worse >= trainerr)

**[B-degree]** Polynomials of degree 0 to 12 fitted to the same forty days: the training error never increases with the degree, while the error on the 326 held-back days is smallest at a low degree and then explodes.

In [ ]:
print("\n[B-degree] the two errors against the degree of the polynomial")
DEGREES = list(range(0, 13))
rows = []
for d in DEGREES:
    coef = np.polyfit(tmin[train], tmax[train], d)
    rows.append((d, avg_sqerr(tmin[train], tmax[train], coef),
                 avg_sqerr(tmin[held], tmax[held], coef)))
    print(f"    degree {d:2d}: training error {rows[-1][1]:8.2f}, "
          f"held-back error {rows[-1][2]:12.2f}")
train_curve = [r[1] for r in rows]
held_curve = [r[2] for r in rows]
check("[B-degree] the training error never increases with the degree",
      all(a >= b - 1e-9 for a, b in zip(train_curve, train_curve[1:])))
check("[B-degree] the training error is below the held-back error at every "
      "degree", all(r[1] < r[2] for r in rows))
best = int(np.argmin(held_curve))
print(f"    smallest held-back error at degree {best}; at degree 12 the "
      f"training error is {train_curve[-1]:.2f} and the held-back error "
      f"{held_curve[-1]:.0f}")
check("[B-degree] the held-back error is smallest at a low degree and grows "
      "by three orders of magnitude beyond degree ten",
      best <= 4 and held_curve[-1] > 1000.0 * min(held_curve))

with open(OUT_DIR / "trainerr_degree.csv", "w") as f:
    f.write("degree,train,heldout\n")
    for d, t, h in rows:
        f.write(f"{d},{t:.4f},{h:.4f}\n")


# ---- preview (checking only)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
ax = axes[0]
ax.plot(grid, np.polyval(quad, grid), "k-", lw=1.4, label="fitted quadratic")
for d in np.sort(six):
    ax.plot([tmin[d], tmin[d]], [np.polyval(quad, tmin[d]), tmax[d]], "k:", lw=1.0)
ax.plot(tmin[pic_train], tmax[pic_train], "o", color="black", ms=7,
        label="training set (4 days)")
ax.plot(tmin[pic_val], tmax[pic_val], "^", mfc="none", mec="black", ms=9,
        mew=1.5, label="held back (2 days)")
ax.set_xlabel("morning minimum temperature (feature, °C)")
ax.set_ylabel("maximum temperature (label, °C)")
ax.set_title(f"Six days at Krems: training error {err_train:.2f}, "
             f"held back {err_val:.2f}", fontsize=9)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax = axes[1]
ax.semilogy(DEGREES, train_curve, "o-", color="black", ms=4,
            label="training error, 40 days")
ax.semilogy(DEGREES, held_curve, "s--", color="0.4", ms=4,
            label="error on the 326 held-back days")
ax.set_xlabel("degree of the polynomial")
ax.set_ylabel("average squared error")
ax.set_title("Fitted by ERM to the same forty days", fontsize=9)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "trainerr.png", dpi=150)
plt.close(fig)

passed = sum(1 for _, ok in report if ok)
print(f"\n{passed}/{len(report)} checks pass")